# Nomadic Intelligence — Parameter-Matched Baseline Experiment

**목적**: Nomadic Full의 성능 우위가 모델 용량(파라미터 수)이 아닌 temporal structure에서 비롯됨을 검증.

## 실험 설계

| 모델 | hidden_dim | 파라미터 수 | 비고 |
|---|---|---|---|
| Fixed MLP (original) | 64 | 4,417 | ablation baseline |
| Standard MoE (original) | 64 | 17,798 | ablation baseline |
| **Fixed MLP (matched)** | **150** | **23,251** | Nomadic Full과 파라미터 동등 |
| **Standard MoE (matched)** | **74** | **23,538** | Nomadic Full과 파라미터 동등 |
| Nomadic NoPolicy | 64 | 22,926 | ablation baseline |
| **Nomadic Full** | **64** | **23,053** | 비교 대상 |

**핵심 가설**: parameter-matched baseline이 Nomadic Full의 Seq MSE에 도달하지 못하거나,
ΔH (entropy differentiation) 패턴을 재현하지 못한다면, 성능 우위는 용량이 아닌
temporal control mechanism(Δx, dwell, PolicyNet)에서 비롯된 것으로 확인.

**Seeds**: 42, 123, 456 | **Epochs**: 220 | **GPU**: L4


In [ ]:
# ============================================================
# STEP 0: 환경 확인
# ============================================================
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA L4
VRAM: 23.7 GB


In [ ]:
# ============================================================
# STEP 1: Imports & Reproducibility
# ============================================================
import os, random, math
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

matplotlib.rcParams['figure.dpi'] = 120

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')


Using device: cuda


In [ ]:
# ============================================================
# STEP 2: Config
# ============================================================
@dataclass
class Config:
    seed: int = 42
    device: str = DEVICE

    # data
    input_dim: int = 2
    output_dim: int = 1
    overlap_std: float = 0.9

    # model (default = original hidden_dim)
    hidden_dim: int = 64
    num_experts: int = 3
    gate_hidden_dim: int = 64
    temperature: float = 0.60

    # training
    epochs: int = 220
    lr: float = 2e-3
    weight_decay: float = 1e-5

    # phase sequence
    phase_batch_size: int = 64
    phase_train_cycles: int = 40
    phase_test_cycles: int = 12
    transition_steps: int = 8

    # hybrid delta
    ema_decay: float = 0.80
    err_baseline_momentum: float = 0.85
    w_env: float = 1.0
    w_err: float = 2.0

    # loss weights
    alpha_dogma: float = 0.04
    beta_nomad: float = 0.05
    beta_phi: float = 0.02        # config.yaml 기준
    gamma_diversity: float = 0.08
    lambda_sep: float = 0.08
    lambda_cons: float = 0.03
    lambda_load: float = 0.03
    tau_k_min: int = 3
    tau_k_penalty: float = 0.05

    # dynamic tau
    use_dynamic_tau: bool = True
    tau_min: float = 2.0
    tau_max: float = 8.0
    tau_var_scale: float = 6.0
    tau_var_window: int = 8

    # phi / switching
    phi_scale_env: float = 1.0
    phi_scale_err: float = 1.5
    phi_scale_explain: float = 1.5   # config.yaml 기준
    phi_scale_gap: float = 0.8       # config.yaml 기준

    temp_stable: float = 0.35        # config.yaml 기준
    temp_transition: float = 0.90    # config.yaml 기준

    use_hard_switch: bool = True
    phi_hard_threshold: float = 0.30  # config.yaml 기준

    # policy
    policy_hidden_dim: int = 64
    policy_mix_weight: float = 0.25
    policy_weight_stay: float = 0.20
    policy_weight_target: float = 0.20
    policy_weight_mode: float = 0.10
    policy_switch_threshold: float = 0.50

    save_dir: str = 'outputs_param_matched'


# ── 실험 조건 정의
# Original hidden_dim=64 / Matched hidden_dim=150(Fixed), 74(StdMoE)
SEEDS = [42, 123, 456]

EXPERIMENT_VARIANTS = [
    # (label, hidden_dim, is_matched)
    ('Fixed_orig',    64,  False),
    ('StdMoE_orig',   64,  False),
    ('Fixed_matched', 150, True),
    ('StdMoE_matched', 74, True),
    ('NoPolicy_orig', 64,  False),
    ('Nomadic_Full',  64,  False),
]

def count_params(model):
    return sum(p.numel() for p in model.parameters())

print('Config loaded.')
print(f'Variants: {[v[0] for v in EXPERIMENT_VARIANTS]}')
print(f'Seeds: {SEEDS}')


Config loaded.
Variants: ['Fixed_orig', 'StdMoE_orig', 'Fixed_matched', 'StdMoE_matched', 'NoPolicy_orig', 'Nomadic_Full']
Seeds: [42, 123, 456]


In [ ]:
# ============================================================
# STEP 3: Data Generation (run_structured.py 동일)
# ============================================================
REGIME_TO_ID = {'A': 0, 'B': 1, 'C': 2}
ID_TO_REGIME = {0: 'A', 1: 'B', 2: 'C'}
REGIME_ORDER = ['A', 'B', 'C']

def sample_regime_x(regime, n, std, device='cpu'):
    noise = std * torch.randn(n, 2, device=device)
    centers = {'A': (2.5, 2.5), 'B': (-2.5, -2.5), 'C': (2.5, -2.5)}
    c = centers[regime]
    return noise + torch.tensor(c, device=device)

def regime_function(x, regime):
    x1, x2 = x[:, 0], x[:, 1]
    if regime == 'A':   y = x1 + x2
    elif regime == 'B': y = x1 - x2
    elif regime == 'C': y = -x1 + 0.5 * x2
    else: raise ValueError(regime)
    return y.unsqueeze(-1)

def generate_phase_sequence(cfg, cycles, device='cpu'):
    xs, ys, rs, phase_tags = [], [], [], []
    for _ in range(cycles):
        for i, curr_r in enumerate(REGIME_ORDER):
            next_r = REGIME_ORDER[(i + 1) % len(REGIME_ORDER)]
            x_s = sample_regime_x(curr_r, cfg.phase_batch_size, cfg.overlap_std, device)
            y_s = regime_function(x_s, curr_r)
            r_s = torch.full((cfg.phase_batch_size,), REGIME_TO_ID[curr_r], dtype=torch.long, device=device)
            xs.append(x_s); ys.append(y_s); rs.append(r_s)
            phase_tags.extend([f'stable_{curr_r}'] * cfg.phase_batch_size)
            for step in range(cfg.transition_steps):
                alpha = (step + 1) / cfg.transition_steps
                x_a = sample_regime_x(curr_r, cfg.phase_batch_size, cfg.overlap_std, device)
                x_b = sample_regime_x(next_r, cfg.phase_batch_size, cfg.overlap_std, device)
                x_mix = (1.0 - alpha) * x_a + alpha * x_b
                y_mix = (1.0 - alpha) * regime_function(x_mix, curr_r) + alpha * regime_function(x_mix, next_r)
                dominant = curr_r if alpha < 0.5 else next_r
                r_mix = torch.full((cfg.phase_batch_size,), REGIME_TO_ID[dominant], dtype=torch.long, device=device)
                xs.append(x_mix); ys.append(y_mix); rs.append(r_mix)
                phase_tags.extend([f'transition_{curr_r}_to_{next_r}'] * cfg.phase_batch_size)
    return torch.cat(xs), torch.cat(ys), torch.cat(rs), phase_tags

def iterate_sequence_minibatches(X, Y, R, batch_size):
    n = X.size(0)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        yield X[start:end], Y[start:end], R[start:end]

print('Data utilities ready.')


Data utilities ready.


In [ ]:
# ============================================================
# STEP 4: Model Definitions (run_structured.py 동일)
# ============================================================
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x): return self.net(x)

class Expert(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x): return self.net(x)

class GateNet(nn.Module):
    def __init__(self, input_dim, gate_hidden_dim, num_experts, use_delta=True):
        super().__init__()
        in_dim = input_dim + 2 if use_delta else input_dim
        self.use_delta = use_delta
        self.net = nn.Sequential(
            nn.Linear(in_dim, gate_hidden_dim), nn.ReLU(),
            nn.Linear(gate_hidden_dim, gate_hidden_dim), nn.ReLU(),
            nn.Linear(gate_hidden_dim, num_experts),
        )
    def forward(self, x, delta_hybrid=None, delta_err=None, temperature=1.0):
        if self.use_delta:
            gate_input = torch.cat([x, delta_hybrid, delta_err], dim=-1)
        else:
            gate_input = x
        logits = self.net(gate_input)
        return F.softmax(logits / temperature, dim=-1), logits

class PolicyNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_experts):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim + 5, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.stay_switch_head = nn.Linear(hidden_dim, 2)
        self.target_head      = nn.Linear(hidden_dim, num_experts)
        self.mode_head        = nn.Linear(hidden_dim, 2)
    def forward(self, policy_input):
        h = self.shared(policy_input)
        return (F.softmax(self.stay_switch_head(h), dim=-1),
                F.softmax(self.target_head(h),      dim=-1),
                F.softmax(self.mode_head(h),        dim=-1))

class StandardMoE(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_experts, gate_hidden_dim):
        super().__init__()
        self.num_experts = num_experts
        self.experts = nn.ModuleList([Expert(input_dim, hidden_dim, output_dim) for _ in range(num_experts)])
        self.gate = nn.Sequential(
            nn.Linear(input_dim, gate_hidden_dim), nn.ReLU(),
            nn.Linear(gate_hidden_dim, gate_hidden_dim), nn.ReLU(),
            nn.Linear(gate_hidden_dim, num_experts),
        )
    def forward(self, x, hard=False):
        logits = self.gate(x)
        gate_probs = F.softmax(logits, dim=-1)
        expert_outputs = torch.stack([e(x) for e in self.experts], dim=1)
        routing = F.one_hot(gate_probs.argmax(-1), self.num_experts).float() if hard else gate_probs
        y_hat = (routing.unsqueeze(-1) * expert_outputs).sum(dim=1)
        return y_hat, gate_probs, logits, expert_outputs

class NomadicMoE(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_experts, gate_hidden_dim, policy_hidden_dim=64):
        super().__init__()
        self.num_experts = num_experts
        self.experts = nn.ModuleList([Expert(input_dim, hidden_dim, output_dim) for _ in range(num_experts)])
        self.gate   = GateNet(input_dim, gate_hidden_dim, num_experts, use_delta=True)
        self.policy = PolicyNet(input_dim, policy_hidden_dim, num_experts)
    def forward(self, x, delta_hybrid, delta_err, temperature, hard=False):
        gate_probs, gate_logits = self.gate(x, delta_hybrid, delta_err, temperature)
        expert_outputs = torch.stack([e(x) for e in self.experts], dim=1)
        routing = F.one_hot(gate_probs.argmax(-1), self.num_experts).float() if hard else gate_probs
        y_hat = (routing.unsqueeze(-1) * expert_outputs).sum(dim=1)
        return y_hat, gate_probs, gate_logits, expert_outputs

# 파라미터 수 확인
cfg_tmp = Config()
fx64  = MLPRegressor(2, 64, 1);  fx150 = MLPRegressor(2, 150, 1)
sm64  = StandardMoE(2, 64, 1, 3, 64);  sm74 = StandardMoE(2, 74, 1, 3, 74)
nm    = NomadicMoE(2, 64, 1, 3, 64, 64)
print(f'Fixed  h=64:  {count_params(fx64):,}')
print(f'Fixed  h=150: {count_params(fx150):,}')
print(f'StdMoE h=64:  {count_params(sm64):,}')
print(f'StdMoE h=74:  {count_params(sm74):,}')
print(f'Nomadic Full: {count_params(nm):,}')
print('\nModel definitions ready.')


Fixed  h=64:  4,417
Fixed  h=150: 23,251
StdMoE h=64:  17,798
StdMoE h=74:  23,538
Nomadic Full: 23,053

Model definitions ready.


In [ ]:
# ============================================================
# STEP 5: Utilities (run_structured.py 동일)
# ============================================================
class HybridDeltaTracker:
    def __init__(self, cfg, device):
        self.cfg = cfg; self.device = device
        self.prev_x_mean = None; self.err_ema = None; self.err_baseline = None
        self.recent_delta_env = deque(maxlen=cfg.tau_var_window)
    def reset(self):
        self.prev_x_mean = None; self.err_ema = None; self.err_baseline = None
        self.recent_delta_env.clear()
    def compute_dynamic_tau(self, sigma2):
        tau = self.cfg.tau_min + (self.cfg.tau_max - self.cfg.tau_min) / (1.0 + self.cfg.tau_var_scale * sigma2)
        return float(np.clip(tau, self.cfg.tau_min, self.cfg.tau_max))
    def compute(self, x, batch_mse):
        x_mean = x.mean(dim=0, keepdim=True)
        de = 0.0 if self.prev_x_mean is None else float(torch.norm(x_mean - self.prev_x_mean, p=2).item())
        batch_err = batch_mse.detach()
        if self.err_ema is None:
            self.err_ema = batch_err; self.err_baseline = batch_err; derr = 0.0
        else:
            self.err_ema = self.cfg.ema_decay * self.err_ema + (1 - self.cfg.ema_decay) * batch_err
            self.err_baseline = self.cfg.err_baseline_momentum * self.err_baseline + (1 - self.cfg.err_baseline_momentum) * self.err_ema
            derr = float(torch.relu(self.err_ema - self.err_baseline).item())
        raw = self.cfg.w_env * de + self.cfg.w_err * derr
        dh = float(torch.tanh(torch.tensor(raw)).item())
        self.prev_x_mean = x_mean.detach()
        self.recent_delta_env.append(de)
        sigma2 = float(np.var(self.recent_delta_env)) if len(self.recent_delta_env) >= 2 else 0.0
        dynamic_tau = self.compute_dynamic_tau(sigma2)
        delta_hybrid = torch.full((x.size(0), 1), dh, device=self.device)
        return delta_hybrid, de, derr, dh, sigma2, dynamic_tau

class DwellTimeRegularizer:
    def __init__(self, tau_k_min=3, penalty=0.05):
        self.tau_k_min = tau_k_min; self.penalty = penalty
        self.current_expert = None; self.dwell_count = 0
    def reset(self):
        self.current_expert = None; self.dwell_count = 0
    def compute(self, gate_probs, tau_dynamic=None):
        dominant = int(torch.bincount(gate_probs.argmax(-1), minlength=gate_probs.size(-1)).argmax().item())
        if dominant == self.current_expert: self.dwell_count += 1
        else: self.current_expert = dominant; self.dwell_count = 1
        eps = 1e-8
        entropy = -(gate_probs * (gate_probs + eps).log()).sum(dim=-1).mean()
        tau_cap = float(tau_dynamic if tau_dynamic is not None else self.tau_k_min)
        if self.dwell_count <= tau_cap:
            return -self.penalty * entropy
        else:
            excess = self.dwell_count - tau_cap
            return min(float(excess) * self.penalty, self.penalty * 10) * entropy

def gate_entropy(gate_probs):
    eps = 1e-8
    return -(gate_probs * (gate_probs + eps).log()).sum(dim=-1)

def compute_load_balancing_loss(gate_probs):
    K = gate_probs.size(-1)
    mean_gate = gate_probs.mean(dim=0)
    top1 = gate_probs.argmax(dim=-1)
    top1_frac = torch.bincount(top1, minlength=K).float() / top1.size(0)
    return K * (top1_frac * mean_gate).sum()

def compute_dogma_penalty(gate_probs):
    mean_usage = gate_probs.mean(dim=0)
    return torch.sum(mean_usage ** 2) - 1.0 / gate_probs.size(1)

def compute_nomad_bonus(gate_probs):
    eps = 1e-8
    return -(gate_probs * (gate_probs + eps).log()).sum(dim=-1).mean()

def compute_diversity_loss(expert_outputs):
    K = expert_outputs.size(1)
    if K < 2: return expert_outputs.new_zeros(1).squeeze()
    idx_i, idx_j = zip(*[(i,j) for i in range(K) for j in range(i+1,K)])
    sim = F.cosine_similarity(expert_outputs[:, idx_i, :], expert_outputs[:, idx_j, :], dim=-1)
    return sim.mean()

def compute_explanation_signals(y_true, y_hat, expert_outputs, gate_probs):
    explanation_error = F.mse_loss(y_hat, y_true)
    per_expert_sqerr = ((expert_outputs - y_true.unsqueeze(1)) ** 2).mean(dim=-1)
    top1_idx = gate_probs.argmax(dim=-1)
    top1_err = per_expert_sqerr.gather(1, top1_idx.unsqueeze(1)).mean()
    best_expert_err = per_expert_sqerr.min(dim=1).values.mean()
    return explanation_error, torch.relu(top1_err - best_expert_err)

def compute_phi_signal(de, derr, explanation_error, best_expert_gap, cfg):
    device = explanation_error.device
    phi = torch.tanh(
        cfg.phi_scale_env     * torch.tensor(de,   device=device)
      + cfg.phi_scale_err     * torch.tensor(derr, device=device)
      + cfg.phi_scale_explain * explanation_error.detach()
      + cfg.phi_scale_gap     * best_expert_gap.detach()
    )
    return phi

def compute_adaptive_temperature(phi_signal, cfg):
    phi_val = float(phi_signal.mean().item())
    return cfg.temp_stable + (cfg.temp_transition - cfg.temp_stable) * phi_val

def build_policy_input(xb, delta_hybrid, delta_err_t, phi_signal, sigma2, dynamic_tau):
    x_summary = xb.mean(dim=0, keepdim=True).expand(xb.size(0), -1)
    phi_t     = torch.full((xb.size(0), 1), float(phi_signal.mean().item()), device=xb.device)
    s2_scaled = float(np.tanh(sigma2 * 10.0))
    tau_scaled= float(np.tanh((dynamic_tau - 5.0) / 5.0))
    s2_t  = torch.full((xb.size(0), 1), s2_scaled, device=xb.device)
    tau_t = torch.full((xb.size(0), 1), tau_scaled, device=xb.device)
    return torch.cat([x_summary, delta_hybrid, delta_err_t, phi_t, s2_t, tau_t], dim=-1)

def build_policy_targets(yb, expert_outputs, phi_signal, sigma2, dynamic_tau, cfg):
    per_expert_sqerr = ((expert_outputs - yb.unsqueeze(1)) ** 2).mean(dim=-1)
    target_expert = per_expert_sqerr.mean(dim=0).argmin().long()
    phi_val = float(phi_signal.mean().item())
    should_switch = (phi_val > cfg.policy_switch_threshold) or (sigma2 > 0.05)
    can_fixate    = (phi_val <= cfg.policy_switch_threshold) and (dynamic_tau >= 5.5)
    return (1 if should_switch else 0), target_expert, (1 if can_fixate else 0)

def compute_regime_gate_stats(gate_probs, regime_ids, num_regimes=3):
    device = gate_probs.device
    valid_means, valid_names = [], []
    l_cons = torch.tensor(0.0, device=device); cnt = 0
    for rid in range(num_regimes):
        mask = regime_ids == rid
        if mask.sum() == 0: continue
        g_r = gate_probs[mask]; u_r = g_r.mean(dim=0)
        valid_means.append(u_r); valid_names.append(ID_TO_REGIME[rid])
        l_cons = l_cons + ((g_r - u_r.unsqueeze(0)) ** 2).sum(dim=-1).mean(); cnt += 1
    if cnt > 0: l_cons = l_cons / cnt
    if len(valid_means) < 2:
        return torch.tensor(0.0, device=device), l_cons
    pairwise = [torch.norm(valid_means[i] - valid_means[j], p=2)
                for i in range(len(valid_means)) for j in range(i+1, len(valid_means))]
    return -torch.stack(pairwise).mean(), l_cons

def regimewise_usage(gate_probs, regime_ids, num_experts):
    usage = {}; top1 = gate_probs.argmax(dim=-1)
    for rid in range(3):
        mask = regime_ids == rid; name = ID_TO_REGIME[rid]
        if mask.sum() == 0: usage[name] = np.zeros(num_experts); continue
        counts = torch.bincount(top1[mask], minlength=num_experts).float()
        usage[name] = (counts / counts.sum().clamp_min(1.0)).cpu().numpy()
    return usage

def compute_dwell_times(seq):
    if len(seq) == 0: return []
    dwells = []; cur = seq[0]; run = 1
    for t in range(1, len(seq)):
        if seq[t] == cur: run += 1
        else: dwells.append(run); cur = seq[t]; run = 1
    dwells.append(run); return dwells

def compute_switch_latency(regime_seq, top1_seq, regime_to_expert):
    latencies = []; prev = regime_seq[0] if regime_seq else None
    for t in range(1, len(regime_seq)):
        curr = regime_seq[t]
        if curr != prev:
            target = regime_to_expert.get(curr)
            if target is not None:
                for k in range(t, len(top1_seq)):
                    if int(top1_seq[k]) == int(target): latencies.append(k - t); break
        prev = curr
    return latencies

def infer_regime_to_expert(usage):
    return {r: int(np.argmax(usage[r])) for r in ['A','B','C']}

print('Utilities ready.')


Utilities ready.


In [ ]:
# ============================================================
# STEP 6: Evaluation Functions
# ============================================================
def eval_fixed(model, X, Y, R, cfg):
    model.eval()
    with torch.no_grad():
        y_pred = model(X)
        seq_mse = F.mse_loss(y_pred, Y).item()
    # seq MSE는 static과 동일 (Fixed는 sequential context 없음)
    return seq_mse

def eval_stdmoe_seq(model, X, Y, R, phase_tags, cfg):
    model.eval()
    all_y, all_gate, batch_tags, batch_ents = [], [], [], []
    with torch.no_grad():
        for bi, (xb, yb, rb) in enumerate(iterate_sequence_minibatches(X, Y, R, cfg.phase_batch_size)):
            y_hat, gate_probs, _, _ = model(xb, hard=False)
            all_y.append(y_hat); all_gate.append(gate_probs)
            batch_tags.append(phase_tags[bi * cfg.phase_batch_size])
            batch_ents.append(gate_entropy(gate_probs).mean().item())
    Y_hat = torch.cat(all_y); G = torch.cat(all_gate)
    seq_mse = F.mse_loss(Y_hat, Y).item()
    stable_h     = [e for t,e in zip(batch_tags, batch_ents) if t.startswith('stable_')]
    transition_h = [e for t,e in zip(batch_tags, batch_ents) if t.startswith('transition_')]
    return seq_mse, {
        'stable_entropy_mean':     float(np.mean(stable_h))     if stable_h     else float('nan'),
        'transition_entropy_mean': float(np.mean(transition_h)) if transition_h else float('nan'),
    }

def eval_nomadic_seq(model, X, Y, R, phase_tags, cfg, use_policy=True):
    model.eval()
    tracker = HybridDeltaTracker(cfg, cfg.device); tracker.reset()
    all_y, all_gate, batch_tags, batch_ents, batch_top1 = [], [], [], [], []
    batch_regimes = []
    with torch.no_grad():
        for bi, (xb, yb, rb) in enumerate(iterate_sequence_minibatches(X, Y, R, cfg.phase_batch_size)):
            z = torch.zeros((xb.size(0), 1), device=cfg.device)
            warm_y, _, _, _ = model(xb, z, z, cfg.temperature)
            warm_mse = F.mse_loss(warm_y, yb)
            delta_hybrid, de, derr, dh, sigma2, dyn_tau = tracker.compute(xb, warm_mse)
            delta_err_t = torch.full((xb.size(0), 1), derr, device=cfg.device)

            probe_y, probe_gate, _, probe_exp = model(xb, delta_hybrid, delta_err_t, cfg.temperature)
            expl_err, gap = compute_explanation_signals(yb, probe_y, probe_exp, probe_gate)
            phi = compute_phi_signal(de, derr, expl_err, gap, cfg)
            temp_now = compute_adaptive_temperature(phi, cfg)

            y_hat, gate_probs, _, exp_out = model(xb, delta_hybrid, delta_err_t, temp_now)

            if use_policy:
                policy_input = build_policy_input(xb, delta_hybrid, delta_err_t, phi, sigma2, dyn_tau)
                stay_sw, tgt_probs, mode_probs = model.policy(policy_input)
                effective_mix = cfg.policy_mix_weight * float(stay_sw[:, 1].mean().item())
                tgt_idx = torch.argmax(tgt_probs.mean(dim=0))
                tgt_oh = F.one_hot(tgt_idx, cfg.num_experts).float().unsqueeze(0).expand(xb.size(0), -1)
                tgt_ste = (tgt_oh - gate_probs).detach() + gate_probs
                mixed = (1.0 - effective_mix) * gate_probs + effective_mix * tgt_ste
                failsafe = dh > cfg.phi_hard_threshold
                hard_mode = cfg.use_hard_switch and (mode_probs[:, 1].mean().item() > 0.5) and not failsafe
                if hard_mode:
                    final_routing = F.one_hot(mixed.argmax(-1), cfg.num_experts).float()
                else:
                    final_routing = mixed
                y_hat = (final_routing.unsqueeze(-1) * exp_out).sum(dim=1)
                gate_probs = final_routing

            all_y.append(y_hat); all_gate.append(gate_probs)
            batch_tags.append(phase_tags[bi * cfg.phase_batch_size])
            batch_ents.append(gate_entropy(gate_probs).mean().item())
            top1 = gate_probs.argmax(-1)
            batch_top1.append(int(torch.bincount(top1, minlength=cfg.num_experts).argmax().item()))
            batch_regimes.append(ID_TO_REGIME[int(rb[0].item())])

    Y_hat = torch.cat(all_y); G = torch.cat(all_gate)
    seq_mse = F.mse_loss(Y_hat, Y).item()
    usage = regimewise_usage(G, R, cfg.num_experts)
    r2e = infer_regime_to_expert(usage)
    latencies = compute_switch_latency(batch_regimes, np.array(batch_top1), r2e)
    stable_h     = [e for t,e in zip(batch_tags, batch_ents) if t.startswith('stable_')]
    transition_h = [e for t,e in zip(batch_tags, batch_ents) if t.startswith('transition_')]
    return seq_mse, {
        'stable_entropy_mean':     float(np.mean(stable_h))     if stable_h     else float('nan'),
        'transition_entropy_mean': float(np.mean(transition_h)) if transition_h else float('nan'),
        'mean_switch_latency':     float(np.mean(latencies))    if latencies    else float('nan'),
    }

print('Evaluation functions ready.')


Evaluation functions ready.


In [ ]:
# ============================================================
# STEP 7: Training Functions
# ============================================================
def train_fixed(cfg, X_train, Y_train, R_train, X_test, Y_test, R_test, phase_tags_test):
    h = cfg.hidden_dim
    model = MLPRegressor(cfg.input_dim, h, cfg.output_dim).to(cfg.device)
    opt   = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    test_seq_mse_log = []
    for epoch in range(cfg.epochs):
        model.train()
        for xb, yb, _ in iterate_sequence_minibatches(X_train, Y_train, R_train, cfg.phase_batch_size):
            opt.zero_grad()
            F.mse_loss(model(xb), yb).backward()
            opt.step()
        seq_mse = eval_fixed(model, X_test, Y_test, R_test, cfg)
        test_seq_mse_log.append(seq_mse)
        if (epoch+1) % 50 == 0 or epoch == 0:
            print(f'  [Fixed h={h}] Ep {epoch+1:03d} | Seq MSE: {seq_mse:.4f}')
    return model, test_seq_mse_log

def train_stdmoe(cfg, X_train, Y_train, R_train, X_test, Y_test, R_test, phase_tags_test):
    h = cfg.hidden_dim
    model = StandardMoE(cfg.input_dim, h, cfg.output_dim, cfg.num_experts, h).to(cfg.device)
    opt   = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    test_seq_mse_log, dyn_log = [], []
    for epoch in range(cfg.epochs):
        model.train()
        for xb, yb, _ in iterate_sequence_minibatches(X_train, Y_train, R_train, cfg.phase_batch_size):
            opt.zero_grad()
            y_hat, gate_probs, _, exp_out = model(xb)
            loss = (F.mse_loss(y_hat, yb)
                    + cfg.gamma_diversity * compute_diversity_loss(exp_out)
                    + cfg.lambda_load     * compute_load_balancing_loss(gate_probs))
            loss.backward(); opt.step()
        seq_mse, dyn = eval_stdmoe_seq(model, X_test, Y_test, R_test, phase_tags_test, cfg)
        test_seq_mse_log.append(seq_mse); dyn_log.append(dyn)
        if (epoch+1) % 50 == 0 or epoch == 0:
            print(f'  [StdMoE h={h}] Ep {epoch+1:03d} | Seq MSE: {seq_mse:.4f} | '
                  f'StableH: {dyn["stable_entropy_mean"]:.4f} | TransH: {dyn["transition_entropy_mean"]:.4f}')
    return model, test_seq_mse_log, dyn_log

def train_nomadic_full(cfg, X_train, Y_train, R_train, X_test, Y_test, R_test, phase_tags_test):
    model = NomadicMoE(cfg.input_dim, cfg.hidden_dim, cfg.output_dim,
                       cfg.num_experts, cfg.gate_hidden_dim, cfg.policy_hidden_dim).to(cfg.device)
    opt   = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    test_seq_mse_log, dyn_log = [], []

    for epoch in range(cfg.epochs):
        model.train()
        tracker  = HybridDeltaTracker(cfg, cfg.device); tracker.reset()
        dwell_reg = DwellTimeRegularizer(cfg.tau_k_min, cfg.tau_k_penalty); dwell_reg.reset()

        for xb, yb, rb in iterate_sequence_minibatches(X_train, Y_train, R_train, cfg.phase_batch_size):
            opt.zero_grad()
            with torch.no_grad():
                z = torch.zeros((xb.size(0),1), device=cfg.device)
                warm_mse = F.mse_loss(model(xb, z, z, cfg.temperature)[0], yb)
            delta_hybrid, de, derr, dh, sigma2, dyn_tau = tracker.compute(xb, warm_mse)
            delta_err_t = torch.full((xb.size(0),1), derr, device=cfg.device)

            with torch.no_grad():
                probe_y, probe_gate, _, probe_exp = model(xb, delta_hybrid, delta_err_t, cfg.temperature)
            expl_err, gap = compute_explanation_signals(yb, probe_y, probe_exp, probe_gate)
            phi = compute_phi_signal(de, derr, expl_err, gap, cfg)
            temp_now = compute_adaptive_temperature(phi, cfg)

            policy_input = build_policy_input(xb, delta_hybrid, delta_err_t, phi, sigma2, dyn_tau)
            stay_sw, tgt_probs, mode_probs = model.policy(policy_input)

            y_hat, gate_probs, _, exp_out = model(xb, delta_hybrid, delta_err_t, temp_now)
            effective_mix = cfg.policy_mix_weight * float(stay_sw[:,1].mean().item())
            tgt_idx = torch.argmax(tgt_probs.mean(0))
            tgt_oh  = F.one_hot(tgt_idx, cfg.num_experts).float().unsqueeze(0).expand(xb.size(0),-1)
            tgt_ste = (tgt_oh - gate_probs).detach() + gate_probs
            mixed   = (1.0 - effective_mix)*gate_probs + effective_mix*tgt_ste
            failsafe = dh > cfg.phi_hard_threshold
            hard_mode = cfg.use_hard_switch and (mode_probs[:,1].mean().item()>0.5) and not failsafe
            final_routing = F.one_hot(mixed.argmax(-1), cfg.num_experts).float() if hard_mode else mixed
            y_hat = (final_routing.unsqueeze(-1) * exp_out).sum(1)

            _, gap_loss = compute_explanation_signals(yb, y_hat, exp_out, final_routing)
            sep_loss, cons_loss = compute_regime_gate_stats(final_routing, rb)
            tau_dwell = dyn_tau if cfg.use_dynamic_tau else float(cfg.tau_k_min)
            dwell_bonus = dwell_reg.compute(final_routing, tau_dynamic=tau_dwell)

            sw_lbl, tgt_lbl, mode_lbl = build_policy_targets(yb, probe_exp, phi, sigma2, dyn_tau, cfg)
            stay_t  = torch.full((xb.size(0),), sw_lbl,               dtype=torch.long, device=cfg.device)
            tgt_t   = torch.full((xb.size(0),), int(tgt_lbl.item()),  dtype=torch.long, device=cfg.device)
            mode_t  = torch.full((xb.size(0),), mode_lbl,             dtype=torch.long, device=cfg.device)

            loss = (F.mse_loss(y_hat, yb)
                    + cfg.beta_phi        * (phi.detach() * gap_loss)
                    + cfg.alpha_dogma     * compute_dogma_penalty(final_routing)
                    - cfg.beta_nomad      * compute_nomad_bonus(final_routing)
                    + cfg.gamma_diversity * compute_diversity_loss(exp_out)
                    + cfg.lambda_sep      * sep_loss
                    + cfg.lambda_cons     * cons_loss
                    + cfg.lambda_load     * compute_load_balancing_loss(final_routing)
                    + cfg.policy_weight_stay   * F.nll_loss(torch.log(stay_sw   + 1e-8), stay_t)
                    + cfg.policy_weight_target * F.nll_loss(torch.log(tgt_probs + 1e-8), tgt_t)
                    + cfg.policy_weight_mode   * F.nll_loss(torch.log(mode_probs+ 1e-8), mode_t)
                    - dwell_bonus)
            loss.backward(); opt.step()

        seq_mse, dyn = eval_nomadic_seq(model, X_test, Y_test, R_test, phase_tags_test, cfg, use_policy=True)
        test_seq_mse_log.append(seq_mse); dyn_log.append(dyn)
        if (epoch+1) % 50 == 0 or epoch == 0:
            print(f'  [Nomadic Full] Ep {epoch+1:03d} | Seq MSE: {seq_mse:.4f} | '
                  f'StableH: {dyn["stable_entropy_mean"]:.4f} | TransH: {dyn["transition_entropy_mean"]:.4f} | '
                  f'Latency: {dyn["mean_switch_latency"]:.4f}')
    return model, test_seq_mse_log, dyn_log

print('Training functions ready.')


Training functions ready.


In [ ]:
# ============================================================
# STEP 8: 실험 실행 — 6종 x 3 seeds
# ============================================================
import time

all_results = {}   # all_results[variant][seed] = {'seq_mse_log', 'dyn_log'}

variant_cfgs = {
    'Fixed_orig':     {'hidden_dim': 64},
    'StdMoE_orig':    {'hidden_dim': 64},
    'Fixed_matched':  {'hidden_dim': 150},
    'StdMoE_matched': {'hidden_dim': 74},
    'NoPolicy_orig':  {'hidden_dim': 64},
    'Nomadic_Full':   {'hidden_dim': 64},
}

for variant, v_cfg in variant_cfgs.items():
    all_results[variant] = {}
    for seed in SEEDS:
        t0 = time.time()
        print(f'\n>>> {variant} | seed={seed}')
        set_seed(seed)
        cfg = Config(seed=seed, hidden_dim=v_cfg['hidden_dim'])
        X_tr, Y_tr, R_tr, tags_tr = generate_phase_sequence(cfg, cfg.phase_train_cycles, cfg.device)
        X_te, Y_te, R_te, tags_te = generate_phase_sequence(cfg, cfg.phase_test_cycles,  cfg.device)

        if variant in ('Fixed_orig', 'Fixed_matched'):
            _, mse_log = train_fixed(cfg, X_tr, Y_tr, R_tr, X_te, Y_te, R_te, tags_te)
            all_results[variant][seed] = {'seq_mse_log': mse_log, 'dyn_log': None}

        elif variant in ('StdMoE_orig', 'StdMoE_matched'):
            cfg.gate_hidden_dim = v_cfg['hidden_dim']   # gate hidden도 맞춤
            _, mse_log, dyn_log = train_stdmoe(cfg, X_tr, Y_tr, R_tr, X_te, Y_te, R_te, tags_te)
            all_results[variant][seed] = {'seq_mse_log': mse_log, 'dyn_log': dyn_log}

        elif variant == 'NoPolicy_orig':
            # Nomadic Full 모델 구조 그대로, policy bypass
            model = NomadicMoE(cfg.input_dim, cfg.hidden_dim, cfg.output_dim,
                               cfg.num_experts, cfg.gate_hidden_dim, cfg.policy_hidden_dim).to(cfg.device)
            opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
            mse_log, dyn_log = [], []
            for epoch in range(cfg.epochs):
                model.train()
                tracker   = HybridDeltaTracker(cfg, cfg.device); tracker.reset()
                dwell_reg = DwellTimeRegularizer(cfg.tau_k_min, cfg.tau_k_penalty); dwell_reg.reset()
                for xb, yb, rb in iterate_sequence_minibatches(X_tr, Y_tr, R_tr, cfg.phase_batch_size):
                    opt.zero_grad()
                    with torch.no_grad():
                        z = torch.zeros((xb.size(0),1), device=cfg.device)
                        warm_mse = F.mse_loss(model(xb, z, z, cfg.temperature)[0], yb)
                    delta_hybrid, de, derr, dh, sigma2, dyn_tau = tracker.compute(xb, warm_mse)
                    delta_err_t = torch.full((xb.size(0),1), derr, device=cfg.device)
                    with torch.no_grad():
                        probe_y, probe_gate, _, probe_exp = model(xb, delta_hybrid, delta_err_t, cfg.temperature)
                    expl_err, gap = compute_explanation_signals(yb, probe_y, probe_exp, probe_gate)
                    phi = compute_phi_signal(de, derr, expl_err, gap, cfg)
                    temp_now = compute_adaptive_temperature(phi, cfg)
                    y_hat, gate_probs, _, exp_out = model(xb, delta_hybrid, delta_err_t, temp_now)
                    final_routing = gate_probs
                    y_hat = (final_routing.unsqueeze(-1) * exp_out).sum(1)
                    _, gap_loss = compute_explanation_signals(yb, y_hat, exp_out, final_routing)
                    sep_loss, cons_loss = compute_regime_gate_stats(final_routing, rb)
                    tau_dwell = dyn_tau if cfg.use_dynamic_tau else float(cfg.tau_k_min)
                    dwell_bonus = dwell_reg.compute(final_routing, tau_dynamic=tau_dwell)
                    loss = (F.mse_loss(y_hat, yb)
                            + cfg.beta_phi        * (phi.detach() * gap_loss)
                            + cfg.alpha_dogma     * compute_dogma_penalty(final_routing)
                            - cfg.beta_nomad      * compute_nomad_bonus(final_routing)
                            + cfg.gamma_diversity * compute_diversity_loss(exp_out)
                            + cfg.lambda_sep      * sep_loss
                            + cfg.lambda_cons     * cons_loss
                            + cfg.lambda_load     * compute_load_balancing_loss(final_routing)
                            - dwell_bonus)
                    loss.backward(); opt.step()
                seq_mse, dyn = eval_nomadic_seq(model, X_te, Y_te, R_te, tags_te, cfg, use_policy=False)
                mse_log.append(seq_mse); dyn_log.append(dyn)
                if (epoch+1) % 50 == 0 or epoch == 0:
                    print(f'  [NoPolicy] Ep {epoch+1:03d} | Seq MSE: {seq_mse:.4f} | '
                          f'StableH: {dyn["stable_entropy_mean"]:.4f}')
            all_results[variant][seed] = {'seq_mse_log': mse_log, 'dyn_log': dyn_log}

        elif variant == 'Nomadic_Full':
            _, mse_log, dyn_log = train_nomadic_full(cfg, X_tr, Y_tr, R_tr, X_te, Y_te, R_te, tags_te)
            all_results[variant][seed] = {'seq_mse_log': mse_log, 'dyn_log': dyn_log}

        elapsed = time.time() - t0
        final_mse = all_results[variant][seed]['seq_mse_log'][-1]
        print(f'    → Final Seq MSE: {final_mse:.4f}  ({elapsed:.0f}s)')

print('\n=== All experiments complete ===')



>>> Fixed_orig | seed=42
  [Fixed h=64] Ep 001 | Seq MSE: 0.5261
  [Fixed h=64] Ep 050 | Seq MSE: 0.4179
  [Fixed h=64] Ep 100 | Seq MSE: 0.4151
  [Fixed h=64] Ep 150 | Seq MSE: 0.4143
  [Fixed h=64] Ep 200 | Seq MSE: 0.4147
    → Final Seq MSE: 0.4137  (332s)

>>> Fixed_orig | seed=123
  [Fixed h=64] Ep 001 | Seq MSE: 0.6268
  [Fixed h=64] Ep 050 | Seq MSE: 0.4061
  [Fixed h=64] Ep 100 | Seq MSE: 0.4039
  [Fixed h=64] Ep 150 | Seq MSE: 0.4044
  [Fixed h=64] Ep 200 | Seq MSE: 0.4032
    → Final Seq MSE: 0.4032  (325s)

>>> Fixed_orig | seed=456
  [Fixed h=64] Ep 001 | Seq MSE: 0.5606
  [Fixed h=64] Ep 050 | Seq MSE: 0.4143
  [Fixed h=64] Ep 100 | Seq MSE: 0.4121
  [Fixed h=64] Ep 150 | Seq MSE: 0.4116
  [Fixed h=64] Ep 200 | Seq MSE: 0.4119
    → Final Seq MSE: 0.4105  (326s)

>>> StdMoE_orig | seed=42
  [StdMoE h=64] Ep 001 | Seq MSE: 0.6096 | StableH: 0.6756 | TransH: 0.8505
  [StdMoE h=64] Ep 050 | Seq MSE: 0.4148 | StableH: 0.9317 | TransH: 0.9397
  [StdMoE h=64] Ep 100 | Seq MSE:

In [ ]:
# ============================================================
# STEP 9: 결과 집계
# ============================================================
rows = []
for variant in variant_cfgs:
    mse_vals, dh_vals, stable_vals, trans_vals = [], [], [], []
    for seed in SEEDS:
        r = all_results[variant][seed]
        mse_vals.append(r['seq_mse_log'][-1])
        if r['dyn_log'] is not None:
            d = r['dyn_log'][-1]
            sh = d['stable_entropy_mean']; th = d['transition_entropy_mean']
            stable_vals.append(sh); trans_vals.append(th)
            dh_vals.append(th - sh if not (math.isnan(th) or math.isnan(sh)) else float('nan'))
        else:
            stable_vals.append(float('nan')); trans_vals.append(float('nan')); dh_vals.append(float('nan'))

    h = variant_cfgs[variant]['hidden_dim']
    rows.append({
        'Variant':       variant,
        'hidden_dim':    h,
        'Seq MSE mean':  np.nanmean(mse_vals),
        'Seq MSE std':   np.nanstd(mse_vals),
        'ΔH mean':       np.nanmean(dh_vals),
        'ΔH std':        np.nanstd(dh_vals),
        'Stable Ent':    np.nanmean(stable_vals),
        'Trans Ent':     np.nanmean(trans_vals),
    })

df = pd.DataFrame(rows)
print('\n' + '='*80)
print('PARAMETER-MATCHED BASELINE EXPERIMENT: Final Results (3-seed mean)')
print('='*80)
print(df.to_string(float_format=lambda x: f'{x:.4f}', index=False))
print('='*80)

# 핵심 비교
nomadic_mse = df[df['Variant']=='Nomadic_Full']['Seq MSE mean'].values[0]
nomadic_dh  = df[df['Variant']=='Nomadic_Full']['ΔH mean'].values[0]
print(f'\nNomadic Full reference: Seq MSE={nomadic_mse:.4f}, ΔH={nomadic_dh:.4f}')
print()
for v in ['Fixed_matched', 'StdMoE_matched']:
    row = df[df['Variant']==v].iloc[0]
    mse_gap = row['Seq MSE mean'] - nomadic_mse
    dh_gap  = row['ΔH mean']      - nomadic_dh
    print(f'{v}: Seq MSE {row["Seq MSE mean"]:.4f} ({mse_gap:+.4f} vs Nomadic) | '
          f'ΔH {row["ΔH mean"]:.4f} ({dh_gap:+.4f} vs Nomadic)')


In [ ]:
# ============================================================
# STEP 10: 시각화 — Seq MSE + ΔH 비교
# ============================================================
COLORS = {
    'Fixed_orig':     '#888780',
    'StdMoE_orig':    '#5DCAA5',
    'Fixed_matched':  '#B4B2A9',
    'StdMoE_matched': '#9FE1CB',
    'NoPolicy_orig':  '#EF9F27',
    'Nomadic_Full':   '#7F77DD',
}
LINESTYLES = {
    'Fixed_orig':     ':',
    'StdMoE_orig':    '--',
    'Fixed_matched':  ':',
    'StdMoE_matched': '--',
    'NoPolicy_orig':  '-.',
    'Nomadic_Full':   '-',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Parameter-Matched Baseline Experiment\n(3-seed mean ± std)', fontsize=12, fontweight='bold')

# Left: Seq MSE curves
ax = axes[0]
for variant in variant_cfgs:
    curves = np.array([all_results[variant][s]['seq_mse_log'] for s in SEEDS])
    mean_c = curves.mean(0); std_c = curves.std(0)
    epochs = np.arange(1, len(mean_c)+1)
    ax.plot(epochs, mean_c, color=COLORS[variant], ls=LINESTYLES[variant], lw=2.0, label=variant)
    ax.fill_between(epochs, mean_c-std_c, mean_c+std_c, color=COLORS[variant], alpha=0.1)
ax.set_xlabel('Epoch'); ax.set_ylabel('Seq MSE'); ax.set_title('Seq MSE (↓ better)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Right: final ΔH bar
ax2 = axes[1]
variants_with_dh = [v for v in variant_cfgs if all_results[v][SEEDS[0]]['dyn_log'] is not None]
dh_means = [np.nanmean([all_results[v][s]['dyn_log'][-1]['transition_entropy_mean']
                         - all_results[v][s]['dyn_log'][-1]['stable_entropy_mean']
                         for s in SEEDS]) for v in variants_with_dh]
dh_stds  = [np.nanstd( [all_results[v][s]['dyn_log'][-1]['transition_entropy_mean']
                         - all_results[v][s]['dyn_log'][-1]['stable_entropy_mean']
                         for s in SEEDS]) for v in variants_with_dh]
x = np.arange(len(variants_with_dh))
ax2.bar(x, dh_means, yerr=dh_stds, capsize=5,
        color=[COLORS[v] for v in variants_with_dh], alpha=0.85, width=0.55)
for i, v in enumerate(variants_with_dh):
    for s in SEEDS:
        d = all_results[v][s]['dyn_log'][-1]
        dh = d['transition_entropy_mean'] - d['stable_entropy_mean']
        ax2.scatter(i, dh, color='black', s=18, zorder=5, alpha=0.7)
ax2.set_xticks(x); ax2.set_xticklabels(variants_with_dh, rotation=15, fontsize=8)
ax2.set_ylabel('ΔH'); ax2.set_title('Final ΔH (↑ better, dots=seeds)')
ax2.axhline(0, color='gray', lw=0.8, ls='--'); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/param_matched_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')


In [ ]:
# ============================================================
# STEP 11: 결과 저장
# ============================================================
import json, os, shutil

SAVE_DIR = '/content/param_matched_results'
os.makedirs(SAVE_DIR, exist_ok=True)

# CSV
df.to_csv(os.path.join(SAVE_DIR, 'results_summary.csv'), index=False)

# per-seed 상세
detail_rows = []
for variant in variant_cfgs:
    for seed in SEEDS:
        r = all_results[variant][seed]
        final_mse = r['seq_mse_log'][-1]
        if r['dyn_log'] is not None:
            d = r['dyn_log'][-1]
            sh = d['stable_entropy_mean']; th = d['transition_entropy_mean']
            lat = d.get('mean_switch_latency', float('nan'))
        else:
            sh = th = lat = float('nan')
        detail_rows.append({
            'Variant': variant, 'Seed': seed, 'hidden_dim': variant_cfgs[variant]['hidden_dim'],
            'Seq MSE': final_mse, 'Stable Ent': sh, 'Trans Ent': th,
            'ΔH': th - sh, 'Switch Lat': lat,
        })

df_detail = pd.DataFrame(detail_rows)
df_detail.to_csv(os.path.join(SAVE_DIR, 'results_per_seed.csv'), index=False)
print(df_detail.to_string(float_format=lambda x: f'{x:.4f}', index=False))

if os.path.exists('/content/param_matched_comparison.png'):
    shutil.copy('/content/param_matched_comparison.png', os.path.join(SAVE_DIR, 'param_matched_comparison.png'))

print(f'\nSaved to: {SAVE_DIR}')
for f in sorted(os.listdir(SAVE_DIR)):
    print(f'  {f}')
